In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

In [2]:
TASK = Path.cwd().parent
ROOT = TASK.parent
OUTPUT = TASK/'output'
CONFIG = json.loads((ROOT/'config.json').read_text())
Y = CONFIG['last_year']

In [3]:
ranks = pd.read_csv(OUTPUT/'rank_correlations.csv')
distributions = pd.read_csv(OUTPUT/'distribution_summary.csv')
distances = pd.read_csv(OUTPUT/'distribution_distances.csv')
moments = pd.read_csv(OUTPUT/'annual_moments.csv')

In [4]:
# common late window; lag-3 and lag-5 pairs fully generated for AR(6,6)
annual = ranks.loc[ranks.measure.eq('annual')]
empirical = annual.loc[annual.source.eq('empirical_complete')]
rows = []
for source in ['empirical_complete'] + CONFIG['models']:
    selected = annual.loc[annual.source.eq(source)]
    row = {'source':source, 'n':int(selected.n.iloc[0]),
           'generated_from_age':6 if source=='ar6_6' else 1 if source!='empirical_complete' else np.nan,
           'fit_cohort':'complete careers' if source=='se_hurdle_s' else 'available observations' if source!='empirical_complete' else 'complete careers'}
    for reference in ['age_0','lag_3','lag_5']:
        row[f'{reference}_rho20'] = selected.loc[selected.reference.eq(reference)&selected.age.eq(Y),'rho'].iloc[0]
        late = selected.loc[selected.reference.eq(reference)&selected.age.between(11,Y)]
        matched = late.merge(empirical.loc[empirical.reference.eq(reference),['age','rho']],on='age',suffixes=('_sim','_emp'))
        row[f'{reference}_rmse11_20'] = np.sqrt(np.mean((matched.rho_sim-matched.rho_emp)**2))
    totals = distributions.loc[distributions.source.eq(source)&distributions.window.eq('career_total')].iloc[0]
    generated = distributions.loc[distributions.source.eq(source)&distributions.window.eq('generated_common')].iloc[0]
    final = moments.loc[moments.source.eq(source)&moments.age.eq(Y)].iloc[0]
    row.update({'mean_career_total':totals.mean_total, 'gini_career_total':totals.gini_total,
                'mean_total6_20':generated.mean_total, 'gini_total6_20':generated.gini_total,
                'final_mean':final['mean'], 'final_zero_fraction':final.zero_fraction})
    if source!='empirical_complete':
        distance = distances.loc[distances.source.eq(source)&distances.diagnostic.eq('generated_common')].iloc[0]
        row['total6_20_ks_D'] = distance.ks_D
    rows.append(row)
comparison = pd.DataFrame(rows)
comparison.to_csv(OUTPUT/'model_comparison.csv',index=False)
display(comparison)

,source,n,generated_from_age,fit_cohort,age_0_rho20,age_0_rmse11_20,lag_3_rho20,lag_3_rmse11_20,lag_5_rho20,lag_5_rmse11_20,mean_career_total,gini_career_total,mean_total6_20,gini_total6_20,final_mean,final_zero_fraction,total6_20_ks_D
0,empirical_complete,510,NaN,complete careers,0.187294,0.000000,0.653156,0.000000,0.582696,0.000000,152.995259,0.360430,111.126608,0.388352,6.789832,0.141176,NaN
1,arw4,10000,1.0,available observations,0.001876,0.226145,0.202827,0.414606,0.077234,0.453604,147.036215,0.183385,104.726944,0.204965,6.666496,0.000000,0.224808
2,lognormal_grw,10000,1.0,available observations,0.126381,0.071691,0.827153,0.190885,0.730615,0.160637,76525.738175,0.986503,76400.881029,0.987112,17713.294556,0.095500,0.413284
3,ar1_s,10000,1.0,available observations,0.001461,0.236766,0.245662,0.358143,0.100119,0.419343,155.612271,0.287743,110.842995,0.345925,6.983392,0.025000,0.068778
4,hurdle_ar1_s_p,10000,1.0,available observations,-0.011916,0.220046,0.308526,0.317960,0.136052,0.391821,135.411339,0.255373,96.931943,0.296480,6.281482,0.156600,0.106145
5,ar6_6,10000,6.0,available observations,0.129372,0.060080,0.592000,0.049308,0.530811,0.034095,136.980226,0.329544,95.644503,0.363410,5.737718,0.164900,0.075892
6,se_hurdle_s,10000,1.0,complete careers,0.159485,0.038154,0.610686,0.043352,0.553039,0.026413,151.488333,0.341177,109.811206,0.377233,7.170356,0.143400,0.046437
